# Etapa 5 · Consultas por tabla de dimensión
## Reto OT Consulting · Soporte analítico para la toma de decisiones

| | |
|---|---|
| **Alumno** | Alexis Arrazola Jiménez · A01664015 |
| **Grupo** | 301 |
| **Corte de los datos** | 17 de noviembre de 2026 |
| **Repositorio** | [github.com/Arrazola77/OTC-EDA-Individual-Alexis](https://github.com/Arrazola77/OTC-EDA-Individual-Alexis) |

Este cuaderno responde la parte A de la Etapa 5: **las consultas de cada tabla de dimensión del modelo dimensional lógico para resolver las preguntas P1 a P11**. Las consultas existen en tres herramientas:

| Herramienta | Archivo en este repositorio | Qué contiene |
|---|---|---|
| **SQL (MySQL 8)** | `00_cargar_datos.sql` y `10_…` a `90_…sql` | El almacén `otc_dw` (9 dimensiones, 3 tablas de hechos y parámetros) y 39 consultas, un archivo por dimensión |
| **Python** | este cuaderno | Carga el almacén y ejecuta las mismas 39 consultas, dimensión por dimensión |
| **Power BI** | `Tablero_etapa_5.pbix` | Tablero con medidas DAX sobre el mismo modelo (utilización, banca, rotación, equipos, habilidades) |

El cuaderno lee los archivos `.sql` del repositorio: **no reescribe las consultas**, las ejecuta tal como están. Para que corra en cualquier equipo, incluido Google Colab, sin instalar un servidor, carga el almacén en SQLite en memoria. Solo traduce dos funciones que SQLite escribe distinto (`CAST(... AS DATE)` y `CONCAT`). Comprobé los resultados contra MySQL (MariaDB 10.11): **37 de 39 consultas dan exactamente lo mismo**. En las otras 2 la diferencia es un redondeo de 534.45 (534.4 contra 534.5) y el orden de dos documentos empatados en 18.

En las salidas **se ocultan los nombres de las personas**; las consultas sí los traen.

In [1]:
import os, re, glob, sqlite3
import pandas as pd
from IPython.display import display, Markdown
pd.set_option('display.width', 200, 'display.max_columns', 30, 'display.max_colwidth', 60)

# Carpeta con los .sql: la del cuaderno, o /content en Colab
CARPETA = next(d for d in ['.', '/content', os.environ.get('OTC_SQL', '.')]
               if os.path.exists(os.path.join(d, '00_cargar_datos.sql')))

def cargar_almacen(ruta_sql):
    # Carga 00_cargar_datos.sql (MySQL) en SQLite en memoria
    txt = open(ruta_sql, encoding='utf-8').read()
    txt = re.sub(r'^\s*(SET|USE|CREATE DATABASE)\b[^;]*;', '', txt, flags=re.M | re.I)
    txt = re.sub(r'^\s*KEY `[^`]+` \([^)]*\),?\n', '', txt, flags=re.M)      # índices secundarios
    txt = re.sub(r'\)\s*ENGINE=[^;]*;', ');', txt)                            # opciones propias de MySQL
    txt = re.sub(r',(\s*\n\s*\);)', r'\1', txt)
    con = sqlite3.connect(':memory:')
    con.executescript(txt)
    return con

def a_sqlite(sql):
    # Traduce las dos funciones de MySQL que SQLite escribe distinto
    sql = re.sub(r'^\s*(USE|SET)\b[^;]*;', '', sql, flags=re.M | re.I)
    sql = re.sub(r'CAST\(\s*(.+?)\s+AS\s+DATE\s*\)', r'DATE(\1)', sql, flags=re.I | re.S)
    sql = re.sub(r'CONCAT\(([^,()]+),\s*([^()]+?)\)', r'(\1 || \2)', sql)
    return sql

def leer_consultas(ruta):
    # Separa un archivo .sql en sus consultas (bloques '-- name:')
    txt = open(ruta, encoding='utf-8').read()
    out = []
    for b in re.split(r'(?m)^-- name: ', txt)[1:]:
        nombre, resto = b.split('\n', 1)
        meta = dict(re.findall(r'(?m)^-- (dimension|pregunta|nota): (.*)$', resto))
        sql = '\n'.join(l for l in resto.splitlines() if not l.startswith('--')).strip()
        out.append({'nombre': nombre.strip(), 'pregunta': meta.get('pregunta', ''), 'nota': meta.get('nota', ''), 'sql': sql})
    return out

OCULTAR = ['nombre_colaborador', 'quienes', 'a_nombre_de', 'pm_otc']
def ejecutar(archivo):
    # Ejecuta todas las consultas de un archivo .sql y muestra cada resultado
    consultas = leer_consultas(os.path.join(CARPETA, archivo))
    for q in consultas:
        df = pd.read_sql(a_sqlite(q['sql']), con)
        for c in OCULTAR:
            if c in df.columns:
                df[c] = df[c].where(df[c].isna() | df[c].isin(['SIN PM']), '(oculto)')
        display(Markdown(f"**`{q['nombre']}`** · {q['pregunta']}" + (f"  \n*Nota: {q['nota']}*" if q['nota'] else '')))
        display(df)
    return len(consultas)

con = cargar_almacen(os.path.join(CARPETA, '00_cargar_datos.sql'))
tablas = pd.read_sql("SELECT name AS tabla FROM sqlite_master WHERE type = 'table' ORDER BY name", con)
tablas['filas'] = [con.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0] for t in tablas.tabla]
tablas['tipo'] = tablas.tabla.str.split('_').str[0].map({'DIM': 'dimensión', 'HECHOS': 'hechos'}).fillna('parámetros')
print('Parámetros:', dict(con.execute('SELECT clave, valor FROM PARAMETROS').fetchall()))
tablas.sort_values(['tipo', 'tabla'])

Parámetros: {'fecha_corte': '2026-11-17', 'horas_por_mes': '160'}


,tabla,filas,tipo
0,DIM_CALIDAD_FECHA,5,dimensión
1,DIM_COLABORADOR,107,dimensión
2,DIM_DEPARTAMENTO,6,dimensión
3,DIM_EQUIPO,123,dimensión
4,DIM_ESTADO_ASIGNACION,5,dimensión
5,DIM_FECHA,3289,dimensión
6,DIM_HABILIDAD,71,dimensión
7,DIM_PROYECTO,12,dimensión
8,DIM_ROL,12,dimensión
9,HECHOS_ASIGNACION,31,hechos


## Mapa: qué dimensión responde cada pregunta

| Pregunta | Dimensión principal | Archivo |
|---|---|---|
| P1. ¿Cuántos activos están asignados y cuántos en banca? | DIM_COLABORADOR | `10_dim_colaborador.sql` |
| P2. ¿Qué asignaciones vencen en 30, 60 y 90 días y quién queda libre? | DIM_ESTADO_ASIGNACION, DIM_FECHA | `30_…`, `40_…` |
| P3. ¿Qué proyectos en curso no tienen personal o PM? | DIM_PROYECTO | `20_dim_proyecto.sql` |
| P4. ¿Cuál es el costo por hora de cada colaborador? | DIM_COLABORADOR, DIM_ROL | `10_…`, `50_…` |
| P5. ¿Cuánto cuesta al mes cada proyecto y cliente? | DIM_PROYECTO, DIM_ROL | `20_…`, `50_…` |
| P6. ¿Cuánto cuesta la banca y de qué tipo es? | DIM_COLABORADOR | `10_dim_colaborador.sql` |
| P7. ¿Qué habilidades tienen menos de 2 personas en nivel Avanzado o Experto? | DIM_HABILIDAD | `80_dim_habilidad.sql` |
| P8. ¿Quién en banca podría cubrir las brechas? | DIM_HABILIDAD, DIM_COLABORADOR | `80_dim_habilidad.sql` |
| P9. ¿Cuál es la rotación mensual y en qué puestos? | DIM_COLABORADOR | `10_dim_colaborador.sql` |
| P10. ¿Cuántos expedientes están completos? | DIM_COLABORADOR | `10_dim_colaborador.sql` |
| P11. ¿Cuántos equipos siguen a nombre de bajas y cuánto valen? | DIM_EQUIPO | `70_dim_equipo.sql` |
| Calidad de datos | DIM_CALIDAD_FECHA, DIM_EQUIPO | `60_…`, `70_…` |
| Contexto organizacional | DIM_DEPARTAMENTO | `90_dim_departamento.sql` |

DIM_EQUIPO, DIM_HABILIDAD y DIM_DEPARTAMENTO, con sus tablas `HECHOS_EQUIPO` y `HECHOS_HABILIDAD`, se agregaron al modelo de la Etapa 4 para cubrir P7, P8, P10 y P11.

## 1. DIM_COLABORADOR · quién
`10_dim_colaborador.sql` · La persona: estatus al corte, puesto, esquema de nómina, costo y expediente. Responde P1, P4, P6, P9 y P10. La banca se calcula como los activos **sin asignación en curso** en `HECHOS_ASIGNACION`.

In [2]:
n = ejecutar('10_dim_colaborador.sql')
print(f'{n} consultas ejecutadas')

**`P1_tasa_utilizacion`** · P1. Que porcentaje de los activos esta asignado a un proyecto y cuantos estan en banca?

,activos,con_asignacion,en_banca,tasa_utilizacion_pct
0,75,30,45,40.0


**`P1_banca_sensibilidad`** · P1 (sensibilidad). Cuantos activos quedan en banca si se excluyen las asignaciones ya vencidas?

,en_banca_ajustada
0,47


**`P1_banca_detalle`** · P1. Quienes estan en banca? (lista para que Direccion les busque proyecto)

,id_colaborador,nombre_colaborador,puesto,familia_puesto,tipo_banca,esquema_nomina,costo_mensual
0,OTC010,(oculto),CONSULTOR FUNCIONAL ORACLE CLOUD Y EBS SENIOR,CONSULTORÍA,BANCA CONSULTORÍA,SMR,85666.88
1,OTC071,(oculto),CONSULTOR TÉCNICO ORACLE CLOUD SENIOR,CONSULTORÍA,BANCA CONSULTORÍA,OTC 100%,82074.30
2,OTC088,(oculto),CONSULTOR FUNCIONAL ORACLE EBS SENIOR,CONSULTORÍA,BANCA CONSULTORÍA,SMR,71800.00
3,OTC075,(oculto),CONSULTOR FUNCIONAL ORACLE EBS SENIOR,CONSULTORÍA,BANCA CONSULTORÍA,SMR,69962.07
4,OTC101,(oculto),CONSULTOR TÉCNICO ORACLE EBS SENIOR,CONSULTORÍA,BANCA CONSULTORÍA,SMR,62262.38
5,OTC034,(oculto),CONSULTOR DBA ORACLE SENIOR,CONSULTORÍA,BANCA CONSULTORÍA,SMR,61533.03
6,OTC038,(oculto),CONSULTOR FUNCIONAL ORACLE CLOUD Y EBS SENIOR,CONSULTORÍA,BANCA CONSULTORÍA,SINDICATO MAGNESON,57604.32
7,OTC069,(oculto),CONSULTOR FUNCIONAL ORACLE EBS SENIOR,CONSULTORÍA,BANCA CONSULTORÍA,SMR,56278.50
8,OTC099,(oculto),CONSULTOR TÉCNICO ORACLE EBS SENIOR,CONSULTORÍA,BANCA CONSULTORÍA,SMR,55485.37
9,OTC039,(oculto),CONSULTOR TÉCNICO RPA SENIOR,CONSULTORÍA,BANCA CONSULTORÍA,SINDICATO MAGNESON,53568.65


... (15 outputs left out)

,activos,expedientes_completos,pct_completos,contratos_pendientes
0,75,41,54.7,61


**`P10_expediente_por_documento`** · P10. Que documentos son los que mas faltan?

,documento,pendientes
0,FOTO,45
1,CONSTANCIA FISCAL,22
2,COMPROBANTE DOMICILIO,21
3,INE,19
4,NSS,18
5,DATOS BANCARIOS,18
6,CURP,17
7,ACTA,16


12 consultas ejecutadas


## 2. DIM_PROYECTO · para quién y en qué
`20_dim_proyecto.sql` · El proyecto y su cliente (jerarquía Cliente > Proyecto). Responde P3 y P5. El costo de proyecto se obtiene sumando la métrica aditiva `costo_mensual_asignado`.

In [3]:
n = ejecutar('20_dim_proyecto.sql')
print(f'{n} consultas ejecutadas')

**`P3_proyectos_en_curso`** · P3. Que proyectos en curso no tienen personal o ya vencieron?

,id_proyecto,cliente,proyecto,estatus_proyecto,fecha_fin,personas_asignadas,sin_personal,vencido,pm_otc
0,C0006-P01,RENT A CENTER (RAC MÉXICO OPERACIONES),SHINE - ASIGNACIÓN DE RECURSOS DE APOYO,EN CURSO,2026-10-30,0,1,1,(oculto)
1,C0004-P01,CITI,SOPORTE ORACLE CLOUD,EN CURSO,NaN,0,1,0,(oculto)
2,C0007-P01,FANOSA,SOPORTE DBA,EN CURSO,NaN,0,1,0,(oculto)
3,C0001-P01,METALSA,AMS MANUFACTURA,EN CURSO,NaN,0,1,0,SIN PM
4,C0002-P01,GRUPO DALTILE,MARAZZI,EN CURSO,2026-09-23,3,0,1,SIN PM
5,C0003-P01,ALLEGION,MIGRACIÓN EBS,EN CURSO,2028-10-30,5,0,0,SIN PM
6,C0001-P02,METALSA,SOPORTE AMS MANUFACTURA,EN CURSO,NaN,5,0,0,SIN PM


**`P3_resumen`** · P3. Cuantos proyectos en curso estan sin personal, cuantos vencidos y cuantos sin PM de OTC?

,proyectos_en_curso,sin_personal,sin_cobertura_pct,vencidos,proyectos_sin_pm_todos
0,7,4,57.0,2,6


**`P5_costo_por_proyecto`** · P5. Cuanto cuesta al mes cada proyecto en plantilla asignada?

,cliente,proyecto,personas,costo_mensual,costo_hora_prom
0,METALSA,ASIGNACIÓN INDEFINIDA,18,1268864.0,458.0
1,METALSA,SOPORTE AMS MANUFACTURA,5,318535.0,398.0
2,GRUPO DALTILE,MARAZZI,3,172382.0,464.0
3,ALLEGION,MIGRACIÓN EBS,5,164929.0,206.0


... (3 outputs left out)

,costo_mensual_asignado_total,asignaciones
0,1924711.0,31


**`P5_costo_por_industria`** · P5 (extra). Cuanto cuesta la plantilla asignada por industria del cliente?

,industria,clientes,costo_mensual
0,AUTOMOTRIZ,1,1587399.0
1,MANUFACTURA,2,337312.0


6 consultas ejecutadas


## 3. DIM_ESTADO_ASIGNACION · en qué situación está
`30_dim_estado_asignacion.sql` · Dimensión de banderas: tipo, estatus, vigencia y si la persona queda libre al vencer. Responde P2.

In [4]:
n = ejecutar('30_dim_estado_asignacion.sql')
print(f'{n} consultas ejecutadas')

**`P2_asignaciones_por_estado`** · P2. Como se reparten las asignaciones segun su vigencia?

,estado_vigencia,tipo_asignacion,persona_libre_al_vencer,asignaciones,personas,costo_mensual
0,INDEFINIDA / SIN FECHA FIN,ASIGNACIÓN,NO APLICA,15,15,1081629.0
1,INDEFINIDA / SIN FECHA FIN,PROYECTO,NO APLICA,8,8,505770.0
2,MÁS DE 90 DÍAS,PROYECTO,SI,5,5,164929.0
3,VENCIDA,PROYECTO,NO,1,1,50467.0
4,VENCIDA,PROYECTO,SI,2,2,121916.0


**`P2_vencen_30_60_90`** · P2. Que asignaciones terminan en los proximos 30, 60 y 90 dias?

,vencidas,vencen_0_30,vencen_31_60,vencen_61_90,mas_de_90,sin_fecha_fin
0,3,0,0,0,5,23


**`P2_personas_libres`** · P2. Quien queda libre cuando termina su asignacion (vencida o por vencer en <= 90 dias)?

,id_asignacion,nombre_colaborador,cliente,proyecto,estado_vigencia,dias_para_fin,costo_mensual_liberado
0,A005,(oculto),GRUPO DALTILE,MARAZZI,VENCIDA,-55.0,85511.0
1,A010,(oculto),GRUPO DALTILE,MARAZZI,VENCIDA,-55.0,36404.0


3 consultas ejecutadas


## 4. DIM_FECHA · cuándo (4 roles)
`40_dim_fecha.sql` · Calendario diario con un miembro DESCONOCIDA (id 0). Se usa como inicio y fin, efectivos y registrados, y como fechas de asignación de equipo. Responde P2 por mes y trimestre.

In [5]:
n = ejecutar('40_dim_fecha.sql')
print(f'{n} consultas ejecutadas')

**`FECHA_rango_calendario`** · Que rango de fechas cubre la dimension de tiempo? (excluye el miembro DESCONOCIDA)

,fecha_mas_antigua,fecha_mas_reciente,dias
0,2019-11-01,2028-10-31,3288


**`FECHA_rango_por_rol`** · Que rango de fechas se usa realmente en cada rol de la tabla de hechos?

,rol,mas_antigua,mas_reciente,con_dato
0,inicio efectivo,2025-11-03,2026-10-13,10
1,fin efectivo,2026-09-23,2028-10-30,8
2,inicio registrado,2025-11-03,2026-10-13,10
3,fin registrado,2027-01-01,2028-07-01,7
4,inicio asignacion de equipo,2019-11-14,2026-10-23,55
5,fin asignacion de equipo,2024-01-22,2026-11-04,10


**`P2_vencimientos_por_mes`** · P2. En que meses terminan las asignaciones y cuanto costo mensual se libera?

,anio,mes,nombre_mes,asignaciones_que_terminan,personas,costo_mensual_que_se_libera
0,2026,9,SEPTIEMBRE,3,3,172382.0
1,2027,7,JULIO,3,3,89856.0
2,2028,7,JULIO,1,1,14024.0
3,2028,10,OCTUBRE,1,1,61049.0


**`P2_inicios_por_trimestre`** · P2 (contexto). Cuando arrancaron las asignaciones con fecha de inicio conocida?

,anio,trimestre,asignaciones_que_inician
0,2025,4,2
1,2026,3,3
2,2026,4,5


4 consultas ejecutadas


## 5. DIM_ROL · con qué función
`50_dim_rol.sql` · El rol con el que se costea la asignación y de dónde salió (asignado, interno o sin rol). Complementa P4 y P5.

In [6]:
n = ejecutar('50_dim_rol.sql')
print(f'{n} consultas ejecutadas')

**`P5_costo_por_rol`** · P5. Cuanto cuesta al mes la plantilla asignada por rol efectivo?

,rol_efectivo,asignaciones,costo_mensual,costo_hora_prom
0,NO APLICA,12,858071.0,447.0
1,CONSULTOR FUNCIONAL ORACLE EBS SENIOR,6,246844.0,257.0
2,CONSULTOR FUNCIONAL ORACLE CLOUD SENIOR,3,197797.0,412.0
3,CONSULTOR TÉCNICO JAVA SENIOR,2,134030.0,419.0
4,CONSULTOR DBA ORACLE SENIOR,1,106050.0,663.0
5,GERENTE DE PROYECTOS Y OPERACIONES,2,100933.0,631.0
6,CONSULTOR TÉCNICO WEB SENIOR,1,73761.0,461.0
7,CONSULTOR FUNCIONAL ORACLE CLOUD Y EBS SENIOR,1,71689.0,448.0
8,CONSULTOR TÉCNICO ORACLE EBS SENIOR,1,61049.0,382.0
9,CONSULTOR KRONOS SENIOR,1,38081.0,238.0


**`P4_origen_del_rol`** · P4/P5. De donde sale el rol con el que se costea cada asignacion?

,fuente_rol,rol_faltante,asignaciones,costo_mensual
0,ROL INTERNO,NO,17,923676.0
1,SIN ROL,SI,12,858071.0
2,ROL ASIGNADO,NO,2,142964.0


2 consultas ejecutadas


## 6. DIM_CALIDAD_FECHA · qué tan confiable es la fecha
`60_dim_calidad_fecha.sql` · Banderas de origen de cada fecha: desconocida, indefinida o ajustada por la regla R08. Mide la calidad de datos.

In [7]:
n = ejecutar('60_dim_calidad_fecha.sql')
print(f'{n} consultas ejecutadas')

**`DQ_calidad_de_fechas`** · Que tan confiables son las fechas de las asignaciones?

,fuente_fecha_inicio,fuente_fecha_fin,fecha_inicio_desconocida,fin_indefinido,ajustada_r08,asignaciones
0,DESCONOCIDA,INDEFINIDA O DESCONOCIDA,SI,SI,NO,21
1,ASIGNACIÓN,ASIGNACIÓN,NO,NO,NO,4
2,PROYECTO (ajustada R08),PROYECTO (ajustada R08),NO,NO,SI,3
3,ASIGNACIÓN,INDEFINIDA O DESCONOCIDA,NO,SI,NO,2
4,ASIGNACIÓN,PROYECTO (completa nulo),NO,SI,NO,1


**`DQ_resumen_fechas`** · Cuantas asignaciones tienen inicio desconocido, fin indefinido o fechas ajustadas por R08?

,inicio_desconocido,fin_indefinido,ajustadas_r08,asignaciones
0,21,24,3,31


2 consultas ejecutadas


## 7. DIM_EQUIPO · equipos de cómputo (complementaria)
`70_dim_equipo.sql` · Responde P11 con `HECHOS_EQUIPO`: situación de cada equipo, equipos a nombre de bajas y su valor conocido.

In [8]:
n = ejecutar('70_dim_equipo.sql')
print(f'{n} consultas ejecutadas')

**`P11_equipos_por_situacion`** · P11. Como se reparten los equipos segun su situacion y cuanto valen?

,situacion_equipo,equipos,valor_conocido,sin_costo
0,EN USO,72,398287.0,35
1,DISPONIBLE,32,69396.0,25
2,DEVUELTO / CERRADO,10,59540.0,5
3,POR RECUPERAR,9,40693.0,5


**`P11_equipos_a_nombre_de_bajas`** · P11. Cuantos equipos siguen a nombre de bajas y cuanto valen?

,equipos_a_nombre_de_bajas,por_recuperar,valor_por_recuperar,por_recuperar_sin_costo
0,17,9,40693.0,5


**`P11_equipos_por_recuperar_detalle`** · P11. Cuales son los equipos por recuperar y a nombre de quien estan?

,cod_activo,tipo,marca,modelo,a_nombre_de,fecha_baja,costo_equipo
0,A0010,LAPTOP,LENOVO,THINKBOOK 14 G2 ITL,(oculto),NaN,12189.0
1,A0034,LAPTOP,DELL,INSPIRON 15 3520,(oculto),NaN,NaN
2,A0047,LAPTOP,DELL,INSPIRON BUSINESS 3520,(oculto),NaN,NaN
3,A0059,LAPTOP,DELL,VOSTRO 3400,(oculto),NaN,NaN
4,A0071,MONITOR,ACER,V226HQL,(oculto),NaN,NaN
5,A0100,LAPTOP,DELL,INSPIRON BUSINESS 3520,(oculto),NaN,8499.0
6,A0030,LAPTOP,DELL,VOSTRO 3400,(oculto),2026-10-30,NaN
7,A0090,LAPTOP,HP,ELITEBOOK 840 G3,(oculto),2026-10-30,7999.0
8,A0026,LAPTOP,DELL,INSPIRON BUSINESS 3520,(oculto),2026-11-09,12006.0


**`DQ_costo_equipos`** · Cuantos equipos no tienen costo registrado? (la inversion real es mayor al valor conocido)

,equipos,sin_costo,pct_sin_costo,inversion_conocida
0,123,70,56.9,567916.0


**`DQ_fecha_equipo_anomala`** · Hay equipos con fechas imposibles? (se tratan como desconocidas)

,cod_activo,tipo,marca,modelo
0,A0075,LAPTOP,HP,PROBOOK 440S


5 consultas ejecutadas


## 8. DIM_HABILIDAD · habilidades (complementaria)
`80_dim_habilidad.sql` · Responde P7 y aproxima P8 con `HECHOS_HABILIDAD`. El cruce exacto de P8 no se puede hacer porque Clientes no trae el perfil que pide cada proyecto.

In [9]:
n = ejecutar('80_dim_habilidad.sql')
print(f'{n} consultas ejecutadas')

**`P7_cobertura_por_habilidad`** · P7. Que habilidades tienen menos de 2 personas activas en nivel Avanzado o Experto?

,codigo_habilidad,habilidad,tecnologia,tipo_habilidad,evaluados_activos,personas_nivel_3_mas
0,H0023,ORACLE EBS - OPM,ORACLE EBS,HARD,10,0
1,H0028,ORACLE EBS - TALEO,ORACLE EBS,HARD,7,0
2,S0004,OTROS - EMPATIA,OTROS,SOFT,0,0
3,S0005,OTROS - MANEJO DE CONFLICTOS,OTROS,SOFT,0,0
4,H0062,PLATAFORMAS TECNOLÓGICAS - FUSE,PLATAFORMAS TECNOLÓGICAS,HARD,8,0
5,H0049,PLATAFORMAS TECNOLÓGICAS - OBI,PLATAFORMAS TECNOLÓGICAS,HARD,13,0
6,H0008,ORACLE EBS - EBTAX,ORACLE EBS,HARD,16,1
7,H0007,ORACLE EBS - XTR,ORACLE EBS,HARD,13,1
8,H0043,ORACLE FUSION CLOUD - LANDED COST,ORACLE FUSION CLOUD,HARD,12,1
9,H0045,ORACLE FUSION CLOUD - MFG,ORACLE FUSION CLOUD,HARD,11,1


**`P7_resumen`** · P7. Cuantas habilidades tienen menos de 2 y cuantas tienen 0 personas activas en nivel 3+?

,habilidades,con_menos_de_2,con_cero
0,71,11,6


**`P8_banca_con_perfil`** · P8. Cuantos de la banca tienen una matriz de habilidades confiable (>= 4 habilidades evaluadas)?

,banca_total,con_matriz_confiable
0,45,24


**`P8_banca_para_cerrar_brechas`** · P8 (aproximacion). Quien de la banca tiene nivel 3+ en las habilidades con cobertura baja?  
*Nota: Clientes no trae el perfil que pide cada proyecto, asi que el cruce exacto con proyectos sin*

,habilidad,personas_nivel_3_mas_hoy,banca_con_nivel_3_mas,quienes
0,ORACLE EBS - EBTAX,1,1,(oculto)
1,ORACLE EBS - XTR,1,1,(oculto)
2,ORACLE FUSION CLOUD - MFG,1,1,(oculto)
3,PLATAFORMAS TECNOLÓGICAS - OCI,1,1,(oculto)


4 consultas ejecutadas


## 9. DIM_DEPARTAMENTO · áreas (complementaria)
`90_dim_departamento.sql` · Plantilla y costo mensual por departamento.

In [10]:
n = ejecutar('90_dim_departamento.sql')
print(f'{n} consultas ejecutadas')

**`DEP_plantilla_y_costo`** · Cuantas personas activas y cuanto cuesta al mes cada departamento?

,departamento,activos,costo_mensual,antiguedad_prom_anios
0,OPERACIONES Y SOPORTE,40,2202761.0,5.2
1,SIN DEPARTAMENTO (SIN PUESTO),29,1630867.0,6.3
2,ADMINISTRACIÓN Y FINANZAS,4,30044.0,7.8
3,VENTAS,1,22499.0,2.1
4,RECURSOS HUMANOS,1,32694.0,1.4


1 consultas ejecutadas


## 10. Valores distintos de cada dimensión
Esta sección prepara la parte B del entregable. Para cada tabla de dimensión cuenta sus miembros y, en cada atributo, los valores distintos:
- Con **10 valores o menos**, los lista.
- Con **más de 10**, los describe: rango si son números o fechas, y los más frecuentes si son texto.

Los nombres de personas y los números de serie no se listan.

In [11]:
NO_LISTAR = {'nombre_colaborador', 'serie', 'pm_otc', 'objetivo'}
pd.set_option('display.max_rows', 100, 'display.max_colwidth', 120)

def entero(x):
    return f'{x:,.0f}' if float(x).is_integer() else f'{x:,.1f}'

def perfil(tabla):
    df = pd.read_sql(f'SELECT * FROM {tabla}', con)
    if tabla == 'DIM_FECHA':
        df = df[df.id_fecha > 0]                       # sin el miembro DESCONOCIDA
    filas = []
    for c in df.columns:
        s = df[c].dropna()
        n = s.nunique()
        num = pd.to_numeric(s, errors='coerce')
        es_num = len(s) > 0 and num.notna().all()
        if es_num and (num % 1 == 0).all():
            s = num.astype('int64')                    # 2019.0 -> 2019
        es_fecha = len(s) > 0 and s.astype(str).str.fullmatch(r'\d{4}-\d{2}-\d{2}').all()
        llave = c.startswith(('id_', 'cod_', 'codigo_')) and n == len(df)
        if c in NO_LISTAR:
            desc = f'{n} textos distintos (no se listan)'
        elif llave and not es_num:
            desc = f'llave: un valor por miembro ({s.min()} a {s.max()})'
        elif llave:
            desc = f'llave: un valor por miembro, de {s.min()} a {s.max()}'
        elif n <= 10:
            desc = '; '.join(str(v) for v in sorted(s.unique()))
        elif es_fecha:
            desc = f'de {s.min()} a {s.max()}'
        elif es_num:
            desc = f'de {entero(num.min())} a {entero(num.max())}; promedio {num.mean():,.1f}'
        elif n == len(s):
            desc = 'un valor por miembro; por ejemplo: ' + ', '.join(s.head(3))
        else:
            top = s.value_counts().head(3)
            desc = 'más frecuentes: ' + ', '.join(f'{k} ({v})' for k, v in top.items())
        filas.append({'atributo': c, 'valores_distintos': n, 'nulos': int(df[c].isna().sum()), 'valores o descripción': desc})
    return len(df), pd.DataFrame(filas)

resumen = []
for t in ['DIM_COLABORADOR', 'DIM_PROYECTO', 'DIM_ROL', 'DIM_FECHA', 'DIM_ESTADO_ASIGNACION',
          'DIM_CALIDAD_FECHA', 'DIM_EQUIPO', 'DIM_HABILIDAD', 'DIM_DEPARTAMENTO']:
    n, p = perfil(t)
    resumen.append({'dimensión': t, 'miembros': n, 'atributos': len(p),
                    'atributos con más de 10 valores': int((p.valores_distintos > 10).sum())})
    display(Markdown(f'### {t} · {n} miembros, {len(p)} atributos'))
    display(p)
pd.DataFrame(resumen)

### DIM_COLABORADOR · 107 miembros, 67 atributos

,atributo,valores_distintos,nulos,valores o descripción
0,id_colaborador,107,0,llave: un valor por miembro (OTC003 a OTC110)
1,nombre_colaborador,107,0,107 textos distintos (no se listan)
2,estatus_origen,2,0,ACTIVO; BAJA
3,estatus_corte,3,0,ACTIVO; ACTIVO (BAJA PROGRAMADA); BAJA
4,es_activo,2,0,0; 1
5,baja_programada,2,0,0; 1
6,fecha_incorporacion,85,1,de 2003-04-01 a 2026-10-06
7,fecha_baja,9,92,2026-08-28; 2026-10-05; 2026-10-12; 2026-10-23; 2026-10-28; 2026-10-29; 2026-10-30; 2026-11-09; 2026-11-19
8,baja_sin_fecha,2,0,0; 1
9,fecha_incorporacion_faltante,2,0,0; 1


### DIM_PROYECTO · 12 miembros, 14 atributos

,atributo,valores_distintos,nulos,valores o descripción
0,id_proyecto,12,0,llave: un valor por miembro (C0001-P01 a C0008-P01)
1,cod_cliente,8,0,C0001; C0002; C0003; C0004; C0005; C0006; C0007; C0008
2,cliente,8,0,ALLEGION; CITI; FANOSA; GRUPO DALTILE; METALSA; NOVOCAST; PARKER SERVICIOS DE MÉXICO; RENT A CENTER (RAC MÉXICO OPER...
3,industria,4,0,AUTOMOTRIZ; CONSULTORÍA; MANUFACTURA; RENTAS DE EQUIPO
4,proyecto,12,0,"un valor por miembro; por ejemplo: AMS MANUFACTURA, SOPORTE AMS MANUFACTURA, ASIGNACIÓN INDEFINIDA"
5,estatus_proyecto,2,0,EN CURSO; POR INICIAR
6,fecha_inicio,6,6,2021-01-01; 2023-01-01; 2025-06-01; 2026-06-30; 2026-07-15; 2026-10-13
7,fecha_fin,3,9,2026-09-23; 2026-10-30; 2028-10-30
8,sin_fecha_inicio,2,0,0; 1
9,sin_fecha_fin,2,0,0; 1


### DIM_ROL · 12 miembros, 6 atributos

,atributo,valores_distintos,nulos,valores o descripción
0,id_rol,12,0,"llave: un valor por miembro, de 1 a 12"
1,rol_interno,11,0,"más frecuentes: NO APLICA (2), CONSULTOR DBA ORACLE SENIOR (1), CONSULTOR FUNCIONAL ORACLE CLOUD SENIOR (1)"
2,rol_asignado,2,0,CONSULTOR FUNCIONAL ORACLE EBS SENIOR; NO APLICA
3,rol_efectivo,11,0,"más frecuentes: CONSULTOR FUNCIONAL ORACLE EBS SENIOR (2), CONSULTOR DBA ORACLE SENIOR (1), CONSULTOR FUNCIONAL ORAC..."
4,fuente_rol,3,0,ROL ASIGNADO; ROL INTERNO; SIN ROL
5,rol_faltante,2,0,NO; SI


... (9 outputs left out)

,atributo,valores_distintos,nulos,valores o descripción
0,codigo_habilidad,71,0,llave: un valor por miembro (H0002 a S0005)
1,habilidad,71,0,"un valor por miembro; por ejemplo: ORACLE EBS - GL, ORACLE EBS - FSG, ORACLE EBS - SLA"
2,tecnologia,6,0,CERTIFICACIONES; IDIOMAS; ORACLE EBS; ORACLE FUSION CLOUD; OTROS; PLATAFORMAS TECNOLÓGICAS
3,tipo_habilidad,2,0,HARD; SOFT


### DIM_DEPARTAMENTO · 6 miembros, 4 atributos

,atributo,valores_distintos,nulos,valores o descripción
0,codigo_departamento,6,0,llave: un valor por miembro (D0001 a D0006)
1,departamento,6,0,ADMINISTRACIÓN Y FINANZAS; DIRECCIÓN GENERAL; OPERACIONES Y SOPORTE; PROCESOS Y MEJORA CONTINUA; RECURSOS HUMANOS; V...
2,objetivo,6,0,6 textos distintos (no se listan)
3,estatus,1,0,ACTIVO


,dimensión,miembros,atributos,atributos con más de 10 valores
0,DIM_COLABORADOR,107,67,22
1,DIM_PROYECTO,12,14,2
2,DIM_ROL,12,6,3
3,DIM_FECHA,3288,7,5
4,DIM_ESTADO_ASIGNACION,5,5,0
5,DIM_CALIDAD_FECHA,5,6,0
6,DIM_EQUIPO,123,17,3
7,DIM_HABILIDAD,71,4,2
8,DIM_DEPARTAMENTO,6,4,0


In [12]:
# Rango de DIM_FECHA y de cada rol de fecha en las tablas de hechos
display(Markdown('**Rango del calendario (sin el miembro DESCONOCIDA)**'))
display(pd.read_sql("SELECT MIN(fecha) AS mas_antigua, MAX(fecha) AS mas_reciente, COUNT(*) AS dias FROM DIM_FECHA WHERE id_fecha > 0", con))
q = [x for x in leer_consultas(os.path.join(CARPETA, '40_dim_fecha.sql')) if x['nombre'] == 'FECHA_rango_por_rol'][0]
display(Markdown('**Rango usado por cada rol de fecha**'))
pd.read_sql(a_sqlite(q['sql']), con)

**Rango del calendario (sin el miembro DESCONOCIDA)**

,mas_antigua,mas_reciente,dias
0,2019-11-01,2028-10-31,3288


**Rango usado por cada rol de fecha**

,rol,mas_antigua,mas_reciente,con_dato
0,inicio efectivo,2025-11-03,2026-10-13,10
1,fin efectivo,2026-09-23,2028-10-30,8
2,inicio registrado,2025-11-03,2026-10-13,10
3,fin registrado,2027-01-01,2028-07-01,7
4,inicio asignacion de equipo,2019-11-14,2026-10-23,55
5,fin asignacion de equipo,2024-01-22,2026-11-04,10


## Conclusión
Las 39 consultas cubren las 9 tablas de dimensión y responden de P1 a P11. Las cifras coinciden con la Etapa 3:
- 40 % de utilización (30 de 75 activos)
- banca de $1,994,154 al mes, que es 50.9 % de la nómina activa
- 4 de 7 proyectos en curso sin personal
- 11 habilidades con menos de 2 personas en nivel Avanzado o Experto
- 9 equipos por recuperar

La descripción de los valores de cada dimensión, la preparación de los datos desde Firebase y cómo instalar la solución están en [`Etapa5_Dimensiones_datos_y_solucion.md`](Etapa5_Dimensiones_datos_y_solucion.md).